# 03 · Data Quality

This dataset has **no missing values**, so there is no imputation step. The real quality issue is **exact duplicate rows**. We document and de-duplicate, then enforce a schema contract.

In [1]:
# --- project-root bootstrap: portable across VS Code / Jupyter / CLI ---
import os
from pathlib import Path
_p = Path.cwd()
while not (_p / 'config' / 'config.yaml').exists() and _p != _p.parent:
    _p = _p.parent
os.chdir(_p)
print('working dir:', Path.cwd())

working dir: /Users/asfalanoi/app_2026/fraud_detection


In [2]:
import pandas as pd
from fraud.io import read_parquet
from fraud.quality import find_duplicates, assert_schema

df = read_parquet('data/interim/creditcard.parquet')
df = df.astype({**{f'V{i}': 'float64' for i in range(1, 29)},
                'Time': 'float64', 'Amount': 'float64', 'Class': 'int64'})
df.shape

(284807, 31)

In [3]:
# Missing values? (expect 0)
print('total missing cells:', int(df.isna().sum().sum()))

total missing cells: 0


In [4]:
# Exact duplicate rows
dups = find_duplicates(df)
before = len(df)
df_dedup = df.drop_duplicates()
after = len(df_dedup)
print(f'duplicate rows: {dups:,}')
print(f'rows before={before:,}  after dedup={after:,}  removed={before-after:,}')

duplicate rows: 1,081
rows before=284,807  after dedup=283,726  removed=1,081


In [5]:
# Schema contract — fails loudly if columns/dtypes drift
expected = {**{f'V{i}': 'float64' for i in range(1, 29)},
            'Time': 'float64', 'Amount': 'float64', 'Class': 'int64'}
assert_schema(df, expected)
print('schema OK')

schema OK


In [6]:
# Value-range sanity
print('Amount min:', df.Amount.min(), '| max:', df.Amount.max())
assert (df.Amount >= 0).all(), 'negative amounts found!'
print('all amounts non-negative')

Amount min: 0.0 | max: 25691.16
all amounts non-negative


**Note:** de-duplication should happen *before* the train/valid/test split to avoid the same transaction leaking across partitions. **Next:** `04_preprocessing`.